# Phase 4 — Silver Transformation

**Objectifs:**
- [x] Cleaning
- [x] Deduplication
- [x] Type Casting
- [x] Handling Null Values
- [x] Rejected Records

**Livrable:** Silver Layer

**Rôle de la couche Silver:** données **propres, typées, dédoublonnées et fiables**. On applique le plan défini en Phase 1 (`ops.data_dictionary.silver_action`).

Les lignes invalides ne sont **jamais supprimées en silence**: elles vont dans une table **`rejected`** avec leur **motif**.

```
bronze.yellow_trips ──► type casting ──► nulls ──► règles de rejet ──┬──► silver.yellow_trips (valides, dédoublonnées)
                                                                    └──► silver.yellow_trips_rejected (+ motif)
```

Toute la logique est dans **`lakehouse_transforms.py`** (testée en Phase 10). Ce notebook l'applique **étape par étape**.

## 0. Setup

In [1]:
from lakehouse_config import *
from lakehouse_transforms import *
spark = get_spark()
setup_schemas(spark)

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

In [2]:
bronze = spark.table(T_BRONZE_TRIPS)
bronze_count = bronze.count()
print(f"Bronze: {bronze_count:,} lignes")

# Rappel du plan defini en Phase 1
spark.table(T_DATA_DICTIONARY).filter("table_name = 'trips'") \
     .select("column_name", "observations", "silver_action").toPandas()

Bronze: 2,964,624 lignes


,column_name,observations,silver_action
0,VendorID,"Valeurs 1, 2, 6",Renommer vendor_id
1,tpep_pickup_datetime,18 lignes hors janvier 2024 (min 2002-12-31),Renommer pickup_datetime ; rejeter hors periode
2,tpep_dropoff_datetime,870 lignes avec dropoff <= pickup,Renommer dropoff_datetime ; rejeter duree <= 0
3,passenger_count,Nulls (Flex Fare) + 31 465 lignes a 0 ; type long,Cast int ; null -> 1 + flag d'imputation
4,trip_distance,60 371 lignes <= 0 ; max 312 722 miles,Rejeter <= 0 et > 100 miles
5,RatecodeID,Nulls (Flex Fare) + 28 663 lignes a 99 ; type ...,Cast int ; null -> 99 (inconnu)
6,store_and_fwd_flag,Nulls (Flex Fare) ; valeurs Y/N,null -> 'N'
7,PULocationID,Toutes dans 1-265,Renommer pu_location_id ; FK vers zones
8,DOLocationID,Toutes dans 1-265,Renommer do_location_id ; FK vers zones
9,payment_type,0 = Flex Fare (= lignes avec nulls) ; type long,Cast int ; garder 0


## 1. Type Casting (+ renommage snake_case)

| Avant (Bronze) | Après (Silver) |
|---|---|
| `VendorID` long/int | `vendor_id` **int** |
| `passenger_count`, `RatecodeID`, `payment_type` **long** | **int** |
| Montants **double** | **decimal(10,2)** (précision monétaire exacte) |
| `tpep_pickup_datetime` timestamp_ntz | `pickup_datetime` **timestamp** |

In [3]:
step1 = standardize_trips(bronze)
step1.printSchema()

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: decimal(10,2) (nullable = true)
 |-- extra: decimal(10,2) (nullable = true)
 |-- mta_tax: decimal(10,2) (nullable = true)
 |-- tip_amount: decimal(10,2) (nullable = true)
 |-- tolls_amount: decimal(10,2) (nullable = true)
 |-- improvement_surcharge: decimal(10,2) (nullable = true)
 |-- total_amount: decimal(10,2) (nullable = true)
 |-- congestion_surcharge: decimal(10,2) (nullable = true)
 |-- airport_fee: decimal(10,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timest

## 2. Handling Null Values
Les nulls concernent **les mêmes 140 162 lignes** (payment_type = 0, Flex Fare). On **impute** une valeur par défaut et on garde un **flag** pour ne pas perdre l'information.

In [4]:
null_cols = ["passenger_count", "ratecode_id", "store_and_fwd_flag", "congestion_surcharge", "airport_fee"]

def null_counts(df):
    return df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in null_cols]).toPandas()

step2 = handle_nulls(step1)
before, after = null_counts(step1), null_counts(step2)
pd.DataFrame({
    "nulls_avant": before.iloc[0], "nulls_apres": after.iloc[0],
    "strategie": ["-> 1 + flag passenger_count_imputed", "-> 99 (code TLC 'inconnu')", "-> 'N'", "-> 0", "-> 0"],
})

,nulls_avant,nulls_apres,strategie
passenger_count,140162,0,-> 1 + flag passenger_count_imputed
ratecode_id,140162,0,-> 99 (code TLC 'inconnu')
store_and_fwd_flag,140162,0,-> 'N'
congestion_surcharge,140162,0,-> 0
airport_fee,140162,0,-> 0


In [5]:
step2.groupBy("passenger_count_imputed").count().show()

+-----------------------+-------+
|passenger_count_imputed|  count|
+-----------------------+-------+
|                  false|2824462|
|                   true| 140162|
+-----------------------+-------+



## 3. Cleaning — règles de rejet

| Motif | Règle |
|---|---|
| `missing_timestamp` | pickup ou dropoff nul |
| `non_positive_duration` | dropoff ≤ pickup |
| `duration_over_24h` | durée > 24 h |
| `non_positive_distance` | distance ≤ 0 |
| `distance_too_large` | distance > 100 miles |
| `negative_amount` | `fare_amount` < 0 ou `total_amount` < 0 |
| `invalid_location` | zone hors 1–265 |
| `pickup_outside_file_period` | mois du pickup ≠ mois du fichier source |

Une ligne peut avoir **plusieurs motifs** (séparés par `; `).

In [6]:
step3 = add_reject_reason(step2)

valid_before_dedup = step3.filter(F.col("reject_reason") == "")
rejected = step3.filter(F.col("reject_reason") != "").withColumn("_rejected_at", F.current_timestamp())

n_valid_before_dedup, n_rejected = valid_before_dedup.count(), rejected.count()
print(f"Valides  : {n_valid_before_dedup:,}")
print(f"Rejetees : {n_rejected:,} ({n_rejected / bronze_count:.2%})")

Valides  : 2,869,912
Rejetees : 94,712 (3.19%)


In [7]:
# Repartition des motifs de rejet (une ligne peut en avoir plusieurs)
(rejected
    .withColumn("reason", F.explode(F.split("reject_reason", "; ")))
    .groupBy("reason").count()
    .withColumn("pct_bronze", F.round(F.col("count") / bronze_count * 100, 3))
    .orderBy(F.desc("count"))
    .show(truncate=False))

+--------------------------+-----+----------+
|reason                    |count|pct_bronze|
+--------------------------+-----+----------+
|non_positive_distance     |60371|2.036     |
|negative_amount           |37568|1.267     |
|non_positive_duration     |870  |0.029     |
|distance_too_large        |59   |0.002     |
|pickup_outside_file_period|18   |0.001     |
|duration_over_24h         |16   |0.001     |
+--------------------------+-----+----------+



## 4. Deduplication
On calcule une **clé technique** `trip_id` = hash SHA-256 des colonnes qui identifient une course. Deux lignes avec le même `trip_id` sont des doublons → on n'en garde qu'une.

In [8]:
step4 = add_trip_id(valid_before_dedup)
silver_trips = enrich_trips(step4).drop("reject_reason").dropDuplicates(["trip_id"])

n_silver = silver_trips.count()
n_duplicates = n_valid_before_dedup - n_silver
print(f"Doublons supprimes: {n_duplicates:,}")
print(f"Silver final      : {n_silver:,}")

Doublons supprimes: 0
Silver final      : 2,869,912


### Colonnes enrichies
`trip_duration_min`, `pickup_date`, `pickup_hour` → préparent les agrégations Gold.

In [9]:
silver_trips.select("trip_id", "pickup_datetime", "trip_duration_min", "pickup_date", "pickup_hour",
                    "trip_distance", "total_amount").show(5, truncate=20)

+--------------------+-------------------+-----------------+-----------+-----------+-------------+------------+
|             trip_id|    pickup_datetime|trip_duration_min|pickup_date|pickup_hour|trip_distance|total_amount|
+--------------------+-------------------+-----------------+-----------+-----------+-------------+------------+
|192714b9146d35d23...|2024-01-01 00:57:55|             19.8| 2024-01-01|          0|         1.72|       22.70|
|782b6d1b708db0ef0...|2024-01-01 00:03:00|              6.6| 2024-01-01|          0|          1.8|       18.75|
|76402296fc79e5dee...|2024-01-01 00:17:06|            17.92| 2024-01-01|          0|          4.7|       31.30|
|d592c7eea26dd41a8...|2024-01-01 00:36:38|              8.3| 2024-01-01|          0|          1.4|       17.00|
|124c1bea933b9e88c...|2024-01-01 00:46:51|              6.1| 2024-01-01|          0|          0.8|       16.10|
+--------------------+-------------------+-----------------+-----------+-----------+-------------+------

## 5. Rejected Records & écriture Silver

In [10]:
(silver_trips.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T_SILVER_TRIPS))

(add_trip_id(rejected).write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T_SILVER_REJECTED))

silver_zones = clean_zones(spark.table(T_BRONZE_ZONES))
silver_zones.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_SILVER_ZONES)

spark.sql(f"COMMENT ON TABLE {T_SILVER_TRIPS} IS 'Silver - courses valides, typees, dedoublonnees (cle trip_id)'")
spark.sql(f"COMMENT ON TABLE {T_SILVER_REJECTED} IS 'Silver - courses rejetees avec motif (reject_reason)'")
spark.sql(f"COMMENT ON TABLE {T_SILVER_ZONES} IS 'Silver - zones TLC normalisees'")
print("Tables Silver ecrites.")

Tables Silver ecrites.


In [11]:
spark.table(T_SILVER_ZONES).filter("location_id IN (1, 132, 264, 265)").show()

+-----------+-------+--------------+------------+
|location_id|borough|          zone|service_zone|
+-----------+-------+--------------+------------+
|          1|    EWR|Newark Airport|         EWR|
|        132| Queens|   JFK Airport|    Airports|
|        264|Unknown|       Unknown|     Unknown|
|        265|Unknown|Outside of NYC|     Unknown|
+-----------+-------+--------------+------------+



## 6. Vérification — réconciliation Bronze = Silver + Rejected + Doublons

In [12]:
s = spark.table(T_SILVER_TRIPS).count()
r = spark.table(T_SILVER_REJECTED).count()
d = n_duplicates

pd.DataFrame([
    ("Bronze", bronze_count),
    ("Silver (valides)", s),
    ("Rejetees", r),
    ("Doublons supprimes", d),
    ("Silver + Rejetees + Doublons", s + r + d),
    ("Reconciliation OK", bronze_count == s + r + d),
], columns=["etape", "lignes"])

,etape,lignes
0,Bronze,2964624
1,Silver (valides),2869912
2,Rejetees,94712
3,Doublons supprimes,0
4,Silver + Rejetees + Doublons,2964624
5,Reconciliation OK,True


In [13]:
# Controles qualite rapides sur Silver
spark.sql(f"""
    SELECT COUNT(*)                                   AS lignes,
           COUNT(DISTINCT trip_id)                    AS trip_id_uniques,
           SUM(CASE WHEN total_amount < 0 THEN 1 END) AS montants_negatifs,
           SUM(CASE WHEN trip_distance <= 0 THEN 1 END) AS distances_invalides,
           MIN(pickup_datetime) AS min_pickup, MAX(pickup_datetime) AS max_pickup
    FROM {T_SILVER_TRIPS}
""").show()

+-------+---------------+-----------------+-------------------+-------------------+-------------------+
| lignes|trip_id_uniques|montants_negatifs|distances_invalides|         min_pickup|         max_pickup|
+-------+---------------+-----------------+-------------------+-------------------+-------------------+
|2869912|        2869912|             NULL|               NULL|2024-01-01 00:00:00|2024-01-31 23:59:55|
+-------+---------------+-----------------+-------------------+-------------------+-------------------+



## Conclusion — Silver Layer

- **`silver.yellow_trips`**: courses **valides**, typées (`int`, `decimal(10,2)`, `timestamp`), en `snake_case`, **sans doublon** (clé `trip_id`) et enrichies (durée, date, heure).
- **`silver.yellow_trips_rejected`**: toutes les lignes écartées **avec leur motif** → rien n'est perdu, on peut auditer et ajuster les règles.
- **`silver.taxi_zones`**: zones normalisées (`N/A` → `Unknown`).
- **Réconciliation exacte:** `Bronze = Silver + Rejetées + Doublons`.
- Principaux motifs de rejet: **distance ≤ 0** et **montants négatifs** (remboursements), conformément à l'analyse de la Phase 1.